In [85]:
import pandas as pd
from openpyxl import load_workbook
from pathlib import Path

In [86]:
# I/O
PATH_RAW_REPORTE_GENERAL = "../raw_data/ReporteGeneral.xls"
PATH_RAW_SIN_FIRMAR_CONTRATO = "../raw_data/SinFirmarContrato.xls"
PATH_RAW_PRUEBAS_INGLES = "../raw_data/Reporte prueba ingles.xlsx"
PATH_CEDULAS_SPRING_2026 = "../raw_data/CI_spring2026.txt"


DIRECTORY_SIN_FIRMAR_CONTRATO = "SinFirmarContrato"
DIRECTORY_INSCRITOS = "InscritosSpring2027"
DIRECTORY_PRUEBAS_POR_AGENDAR = "PruebasPorAgendar"
COLUMNAS_PROCESOS = ['CI', 'Celular','Nombre', 'Estatus Participante', 'Sponsor','Empleador', 'Posicion Laboral', 'Correo', 'Fecha Inscripcion', 'Precio']

In [87]:
def get_active_profiles(df: pd.DataFrame, JustRetired: bool) -> pd.DataFrame:

    if not isinstance(df, pd.DataFrame):
        raise TypeError(f"Se esperara recibir un DataFrame, se recibio {type(df)}")
    if df.empty:
        raise ValueError(f"Se está recibiendo un DataFrame vacío")
    
    if JustRetired:
        mask_activos = df['Estatus Participante'] == "Activo"
    else:
        mask_activos = df["Estatus Participante"] != "Retirado"
        
    mask_es_prueba = df['Nombre'].str.lower().str.contains('prueba')
    
    return df.loc[~mask_es_prueba & mask_activos].copy()
    

In [88]:
def is_monday() -> bool:
    today = pd.Timestamp.now().day_name()
    return True if today == "Monday" else False

In [89]:
def get_fecha_corte(today: bool) -> pd.Timestamp:
    if today:
        return pd.Timestamp.now().normalize()
    else:
        if is_monday():
            return (pd.Timestamp.today() -  pd.to_timedelta('3 days')).normalize()
        else:
            return  (pd.Timestamp.today() - pd.to_timedelta('1 day')).normalize()       

In [90]:
def save_report(df: pd.DataFrame, directory: str, file_name: str, today: bool) -> str:
    
    if not isinstance(df, pd.DataFrame):
        raise TypeError(f"Se esperara recibir un DataFrame, se recibio {type(df)}")
    if df.empty:
        raise ValueError(f"Se está recibiendo un DataFrame vacío")
    
    date_report = pd.Timestamp.today().normalize().strftime("%d-%m-%Y") if today else (pd.Timestamp.today() - pd.to_timedelta('1 day')).normalize().strftime("%d-%m-%Y")
    file_name_to_save = file_name + "_" + date_report + ".xlsx"
    path_file = directory + "/" + file_name_to_save
    with pd.ExcelWriter(path_file) as writer:
        df.to_excel(writer, index=False)
    print(f"Reporte {file_name} guardado exitosamete!")
    return path_file


In [91]:
def format_cells(file_path: str) -> None:

    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"Archivo excel no encontrado: {file_path}")
    
    wb = load_workbook(file_path)
    ws = wb.active

    for column in ws.columns:
        max_lenght = max(len(str(cell.value)) for cell in column if cell.value)
        ws.column_dimensions[column[0].column_letter].width =  max_lenght + 2 
    
    wb.save(file_path)
    print("Reporte formateado correctamente!")

In [92]:
def read_excel(excel_path: str, skip_rows: int) -> pd.DataFrame:
    path = Path(excel_path)
    if not path.exists():
        raise FileNotFoundError(f"Archivo excel no encontrado: {excel_path}")
    
    return pd.read_excel(io=path, skiprows=skip_rows)


In [93]:
def find_ex_participantes_ci(ci_2027:set[str], ci_2026:set[str]) -> list[str]:
    return list(ci_2027.intersection(ci_2026))
    


In [94]:
def find_new_participantes_ci(ci_2027:set[str], ci_2026:set[str]) -> list[str]:
  return list(ci_2027.difference(ci_2026))

In [95]:
def read_txt(txt_path) -> set[str]:
    cedulas = set({})
    path = Path(txt_path)
    if not path.exists():
        raise FileNotFoundError(f"Archivo txt no encontrado: {txt_path}")
    with open(txt_path, "r", encoding="utf-8") as file:
        for line in file:
            cedulas.add(line.strip().zfill(10))
    return cedulas

In [96]:
# Cargar cedula ex participantes
try:
    cedulas_spring_2026 = read_txt(PATH_CEDULAS_SPRING_2026)
    print("Cedulas spring 2027 cargadas")
except Exception as e:
    print(F"[ERROR] existio un erro al leer archivo txt => {e}")


Cedulas spring 2027 cargadas


# Reporte General

In [97]:
try:
    df_activos = get_active_profiles(read_excel(PATH_RAW_REPORTE_GENERAL, 6), JustRetired=True)
    df_activos = df_activos[COLUMNAS_PROCESOS]
    print(f"Número de participantes activos => {df_activos.shape[0]}")
except Exception as e:
     print(f"[ERROR] al cargar archivo de entrada => {e}")
     raise

Número de participantes activos => 504


In [98]:
mask_sin_precio_capturado =  df_activos['Precio'].str.split(" ").str[-1] == "0.00"
df_precios_no_capturados = df_activos.loc[mask_sin_precio_capturado].copy()

if df_precios_no_capturados.empty:
    print("No hay precio por capturar")
else:
    print(f"Número de participantes sin precio capturado => {len(df_precios_no_capturados)}")
    display(df_precios_no_capturados)

Número de participantes sin precio capturado => 17


,CI,Celular,Nombre,Estatus Participante,Sponsor,Empleador,Posicion Laboral,Correo,Fecha Inscripcion,Precio
139,1207889997,985164953,Maria de los Angeles Correa Trivino,Activo,LIFETRAVELED,Colorvision-Universal Studios,Sales Associate/Photographer,mariacorreatrivino@gmail.com,2026-05-29,Usuario Actualizo - Precio - 0.00
171,1315143683,994961004,Oliver Aaron Balderramo Loor,Activo,CHI,GRAND PRIX,Ride Attendant,oliversbv20@gmail.com,2026-05-29,Usuario Actualizo - Precio - 0.00
187,941644122,961514593,Israel Sebastian Toral Fiallos,Activo,CHI,GRAND PRIX,Ride Attendant,israeltoral1@hotmail.com,2026-05-29,Usuario Actualizo - Precio - 0.00
210,955340450,995213354,Piero Adonis Cirino Onofre,Activo,SPIRIT,MT. OLYMPUS,Houseperson,adonispiero13@gmail.com,2026-05-29,Usuario Actualizo - Precio - 0.00
287,955349394,939247084,Luiggi Yasmani Montano Veas,Activo,SPIRIT,MT. OLYMPUS,Houseperson,luiggiveas45@gmail.com,2026-05-29,Usuario Actualizo - Precio - 0.00
311,958744310,939505720,Matthews Alexander Cedeno Elizondo,Activo,PENDIENTE POR DESTINO,PENDIENTE POR DESTINO,PENDIENTE POR DESTINO,matthewscedeno9@gmail.com,2026-05-29,Usuario Actualizo - Precio - 0.00
316,953899184,993082614,Geovanny Gabriel Asadobay Jaramillo,Activo,SPIRIT,THE HANGOUT,Back of the House Position,geovannyasadobay2404@gmail.com,2026-05-29,Usuario Actualizo - Precio - 0.00
338,929392538,959010384,Giusseppe Geovanny Chacon Cali,Activo,AAG,Landry's Bubba Gump Shrimp Co - Galveston,Cook,giusseppechacon22@gmail.com,2026-05-29,Usuario Actualizo - Precio - 0.00
380,943799973,982674578,Maria Fernanda Cevallos Villamar,Activo,AAG,Kalahari Resort Round Rock,Restaurant Attendant,mafercevallos301@gmail.com,2026-05-30,Usuario Actualizo - Precio - 0.00
397,926972712,980457510,Charbel Angel Ordonez Mendoza,Activo,CHI,LCP Cupples - Westin St. Louis,Room attendant,charbelordonezm@gmail.com,2026-05-29,Usuario Actualizo - Precio - 0.00


In [99]:
cedulas_srping_2027 = set(df_activos['CI'].astype('str').str.zfill(10))
ci_ex_participantes_en_spring_2027 = find_ex_participantes_ci(ci_2027=cedulas_srping_2027, ci_2026=cedulas_spring_2026)
ci_nuevos_participantes_srping_2027 = find_new_participantes_ci(ci_2027=cedulas_srping_2027, ci_2026=cedulas_spring_2026)
print(f"Número de ex participantes inscritos para srping 2027 => {len(ci_ex_participantes_en_spring_2027)}")
print(f"Número de nuevos participantes inscritos para srping 2027 => {len(ci_nuevos_participantes_srping_2027)}")

Número de ex participantes inscritos para srping 2027 => 120
Número de nuevos participantes inscritos para srping 2027 => 384


## Participantes inscritos ayer

In [100]:
fecha_corte = get_fecha_corte(today=False)
mask_inscritos_ayer = df_activos['Fecha Inscripcion'] == fecha_corte
df_inscritos_ayer = df_activos.loc[mask_inscritos_ayer].copy()
if df_inscritos_ayer.empty:
    print("No hubieron participantes inscritos ayer")
else:
    print(f"Número de inscritos ayer => {len(df_inscritos_ayer)}")
    display(df_inscritos_ayer)

Número de inscritos ayer => 3


,CI,Celular,Nombre,Estatus Participante,Sponsor,Empleador,Posicion Laboral,Correo,Fecha Inscripcion,Precio
380,943799973,982674578,Maria Fernanda Cevallos Villamar,Activo,AAG,Kalahari Resort Round Rock,Restaurant Attendant,mafercevallos301@gmail.com,2026-05-30,Usuario Actualizo - Precio - 0.00
462,955727540,991738320,Pablo Javier Velastegui Vargas,Activo,CHI,GRAND PRIX,Admissions,pvelaste@espol.edu.ec,2026-05-30,Usuario Actualizo - Precio - 0.00
485,705942399,990999404,Priscilla Yazbeth Armijos Solano,Activo,SPIRIT,MT. OLYMPUS,Flex PM,prisciarmijos601@gmail.com,2026-05-30,Usuario Actualizo - Precio - 0.00


In [101]:
try:
    save_path = save_report(df_inscritos_ayer, DIRECTORY_INSCRITOS ,"Inscritos", False)
except Exception as e:
    print(f"[ERROR] al guardar el reporte de excel => {e}")

try:
    format_cells(save_path)
except Exception as e:
    print(f"[ERROR] al formatear celdad reporte excel => {e}")


Reporte Inscritos guardado exitosamete!
Reporte formateado correctamente!


## Inscritos el día de hoy

In [102]:
fecha_corte = get_fecha_corte(today=True)
mask_today = df_activos['Fecha Inscripcion'] == fecha_corte
if df_activos.loc[mask_today].empty:
    print("Hoy no hay inscritos")
else:
    print(f"Número de inscritos el día de hoy {len(df_activos.loc[mask_today])}")
    display(df_activos.loc[mask_today])

Hoy no hay inscritos


# Sin Firmar Contrato

In [103]:
df_sin_firmar_contrato = get_active_profiles(read_excel(PATH_RAW_SIN_FIRMAR_CONTRATO, skip_rows=6), JustRetired=False)
df_sin_firmar_contrato = df_sin_firmar_contrato[COLUMNAS_PROCESOS].sort_values(by="Fecha Inscripcion")
if df_sin_firmar_contrato.empty:
    print("No hay contratos por firmar")
else:
    print(f"Número de participantes sin firmar contrato {len(df_sin_firmar_contrato)}")
    display(df_sin_firmar_contrato)

Número de participantes sin firmar contrato 30


,CI,Celular,Nombre,Estatus Participante,Sponsor,Empleador,Posicion Laboral,Correo,Fecha Inscripcion,Precio
23,1351548811,995392072,Jeremy Alberto Acuna Proano,Activo,CHI,Northwest X Southern Hospitality LLC,Room Attendant,jeremyacuna75@gmail.com,2026-03-04,Usuario Actualizo Melissagomez - Precio - 1650.00
12,2450843723,986624103,Bianca Estephania Bravo Aviles,Activo,CHI,GRAND PRIX,Food and Beverage,biancabravo1412@gmail.com,2026-05-15,Usuario Actualizo TanyaQuiroz - Precio - 1750.00
1,929283315,985934076,Nayvelyn Nahuel Tagle Franco,Activo,SPIRIT,WILDERNESS TENNESSEE,Activities Associate/Mascot Character,naytafra@gmail.com,2026-05-18,Usuario Actualizo RominaLeonCarpio - Precio - ...
0,954429742,978789977,José Antonio Rojas Guerra,Activo,AAG,Great Wolf Lodge - Sandusky,Lifeguard,rojasguerrajoseantonio@gmail.com,2026-05-20,Usuario Actualizo RECEPCIONUIO - Precio - 1949.00
2,954151909,985528396,Gino Leonardo Quinde Espinoza,Activo,AAG,Great Wolf Lodge - Sandusky,Lifeguard,quindegino@gmail.com,2026-05-20,Usuario Actualizo RECEPCIONUIO - Precio - 1949.00
6,926919622,967688326,Luisa Fernanda Toro Oramas,Activo,SPIRIT,THE HANGOUT,Back of the House Position,fernantotoro@gmail.com,2026-05-28,Usuario Actualizo RECEPCIONUIO - Precio - 1949.00
34,1720992468,969428200,Alisson Dayane Ospina Jaramillo,Eliminado,GREENHEART,SC Seaside CO dba Santa Cruz Beach Boardwalk,Rider Operators,alissonospina08@gmail.com,2026-05-29,Usuario Actualizo - Precio - 0.00
32,1314615160,979632371,Lucianna Angelina Rivera Zamora,Activo,CHI,Orbit Inc - I Scream Candy - Broadway At The B...,Ride Operator,Lucianariverazamora@gmail.com,2026-05-29,Usuario Actualizo - Precio - 0.00
31,1754327458,987055281,Maria Belen Medina Bocca,Activo,CHI,GRAND PRIX,Food and Beverage,belen.medinabocca@gmail.com,2026-05-29,Usuario Actualizo - Precio - 0.00
30,707109682,998543642,Saray Gislayne Toapanta Gonzabay,Activo,SPIRIT,MT. OLYMPUS,Lifeguard,toapantasaray13@gmail.com,2026-05-29,Usuario Actualizo - Precio - 0.00


In [104]:
df_sin_firmar_contrato['Fecha Inscripcion'].dt.month_name().value_counts()

Fecha Inscripcion
May      29
March     1
Name: count, dtype: int64

In [105]:
try:
    save_path = save_report(df_sin_firmar_contrato, DIRECTORY_SIN_FIRMAR_CONTRATO ,"SinFirmarContrato", True)
except Exception as e:
    print(f"[ERROR] al guardar el reporte de excel => {e}")

try:
    format_cells(save_path)
except Exception as e:
    print(f"[ERROR] al formatear celdad reporte excel => {e}")

Reporte SinFirmarContrato guardado exitosamete!
Reporte formateado correctamente!


# Prueba de inglés

In [106]:
df_pruebas_ingles = get_active_profiles(read_excel(PATH_RAW_PRUEBAS_INGLES, skip_rows=1), JustRetired=False)
print(f"{len(df_pruebas_ingles)}")

511


In [107]:
mask_oral_tomada = df_pruebas_ingles['Cualitativa'] != "Sin Nota"
mask_sin_fecha_prueba_ingles = df_pruebas_ingles["Fecha agenda prueba"] == "\xa0"

if df_pruebas_ingles.loc[mask_oral_tomada].empty:
    print("No se ha tomado niguna prueba oral de inglés")
else:
    print(f"Número de pruebas orales tomadas => {len(df_pruebas_ingles.loc[mask_oral_tomada])}")

Número de pruebas orales tomadas => 250


In [113]:
if df_pruebas_ingles.loc[~mask_oral_tomada & ~mask_sin_fecha_prueba_ingles].empty:
    print("0 pruebas por tomar")
else:
    print(f"Número de pruebas agendadas => {len(df_pruebas_ingles.loc[~mask_oral_tomada & ~mask_sin_fecha_prueba_ingles])}")

Número de pruebas agendadas => 132


In [109]:

df_no_prueba_ingles_oral =  df_pruebas_ingles.loc[mask_sin_fecha_prueba_ingles & ~mask_oral_tomada]  # Getting a df 
ci_no_prueba_ingles_oral =  df_pruebas_ingles.loc[mask_sin_fecha_prueba_ingles & ~mask_oral_tomada, "CI"].astype("str").str.zfill(10) # Getting Series of srings
ci_nuevos_participantes_sin_prueba = find_new_participantes_ci(ci_2027=set(ci_no_prueba_ingles_oral), ci_2026=cedulas_spring_2026) # Getting just CI
mask_nuevos_participantes_sin_prueba = ci_no_prueba_ingles_oral.isin(ci_nuevos_participantes_sin_prueba)
df_nuevos_por_agendar_prueba_ingles = df_no_prueba_ingles_oral[mask_nuevos_participantes_sin_prueba]

if df_nuevos_por_agendar_prueba_ingles.empty:
    print("No hay participantes para agendar prueba de inglés")
else:
    print(f"Número de pruebas orales de inglés por agendar => {len(df_nuevos_por_agendar_prueba_ingles)}")

Número de pruebas orales de inglés por agendar => 21


In [110]:
try:
    save_path = save_report(df_nuevos_por_agendar_prueba_ingles, DIRECTORY_PRUEBAS_POR_AGENDAR ,"PruebasPorAgendar", True)
except Exception as e:
    print(f"[ERROR] al guardar el reporte de excel => {e}")

try:
    format_cells(save_path)
except Exception as e:
    print(f"[ERROR] al formatear celdad reporte excel => {e}")

Reporte PruebasPorAgendar guardado exitosamete!
Reporte formateado correctamente!


In [111]:
# Si hay "Eliminados" saldrá más que el total de activos
sin_prueba_escrita = df_pruebas_ingles['Calificacion Escrita'] == "Sin Calificacion"
con_prueba_escrita = df_pruebas_ingles['Calificacion Escrita'] != 'Sin Calificacion'
print(f'Participantes SIN prueba de inglés escrita => {sin_prueba_escrita.sum()}')
print(f'Participantes CON prueba de inglés escrita => {con_prueba_escrita.sum()}')

Participantes SIN prueba de inglés escrita => 158
Participantes CON prueba de inglés escrita => 353


In [112]:
df_activos["Fecha Inscripcion"].dt.month_name().value_counts()

Fecha Inscripcion
May         186
April       157
March       118
February     39
January       4
Name: count, dtype: int64

In [115]:
df_activos["Fecha Inscripcion"].dt.year

0      2025
1      2026
2      2026
3      2026
4      2026
       ... 
509    2026
510    2026
511    2026
512    2026
513    2026
Name: Fecha Inscripcion, Length: 504, dtype: int32